# Phase 3 - DuckDB Materialization

This notebook materializes the validated star schema into a DuckDB database.

## Objectives:
- Create (or reuse) a DuckDB database file under `04-duckdb/`
- Load raw CSVs into staging tables (`stg_*`)
- Execute SQL models from `03-sql/` to build dimensions and fact tables
- Validate materialized outputs (row counts, grain integrity)

## Assumptions:
- Phase 1-2 (data validation + logical star schema) is complete
- Raw files exist under `01-data/01-raw/`
- SQL scripts exist under `03-sql/` (e.g, `models/`, `schema/` and `marts/`)

In [1]:
import duckdb
from pathlib import Path
from typing import Optional
con: Optional[duckdb.DuckDBPyConnection] = None


#Resolve project root dynamically (portable across machines and launch directories)
cwd = Path.cwd().resolve()

def find_project_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "01-data").exists() and (p / "02-notebooks").exists():
            return p
    raise FileNotFoundError(
        "Project root not found. Expected folders: '01-data' and '02-notebooks'."
    )

PROJECT_ROOT = find_project_root(cwd)

PROJECT_ROOT

PosixPath('/Users/manuel.orozco/Desktop/SALES ANALYTICS DASHBOARD')

## DuckDB database file

The DuckDB database is stored under `04-duckdb/` as a generated artifact.

If this repository is version-controlled, you typically add `04-duckdb/*.duckdb` to `.gitignore`.

In [2]:
DUCKDB_DIR = PROJECT_ROOT / "04-duckdb"
DUCKDB_DIR.mkdir(parents=True, exist_ok=True)

DUCKDB_PATH = DUCKDB_DIR / "sales_analytics.duckdb"

# If rerunning in the same kernel, close any existing connection first
if con is not None:
    try:
        con.close()
    except Exception:
        pass

con = duckdb.connect(str(DUCKDB_PATH))
DUCKDB_PATH

PosixPath('/Users/manuel.orozco/Desktop/SALES ANALYTICS DASHBOARD/04-duckdb/sales_analytics.duckdb')

In [3]:
con.execute("select version() as duckdb_version;").fetchdf(), con.execute("show tables;").fetchdf()

(  duckdb_version
 0         v1.4.4,
                               name
 0                    dim_customers
 1                         dim_date
 2                     dim_products
 3                 fact_order_items
 4              fact_order_payments
 5               fact_order_reviews
 6                      fact_orders
 7       mart_cohort_unit_economics
 8        mart_customer_ltv_summary
 9   mart_monthly_business_snapshot
 10                   stg_customers
 11                 stg_order_items
 12              stg_order_payments
 13               stg_order_reviews
 14                      stg_orders
 15                    stg_products)

## Staging (raw -> DuckDB)

We load the raw CSV files from `01-data/01-raw` into staging tables (`stg_*`).

Staging tables preserve raw structure and act as the source for SQL models.

In [4]:
RAW_DIR = PROJECT_ROOT / "01-data" / "01-raw"

expected_files = {
    "orders": "orders_dataset.csv",
    "customers": "customers_dataset.csv",
    "order_items": "order_items_dataset.csv",
    "order_payments": "order_payments_dataset.csv",
    "order_reviews": "order_reviews_dataset.csv",
    "products": "product_summarize_dataset.csv",
}

missing = [f for f in expected_files.values() if not (RAW_DIR / f).exists()]
assert not missing, f"Missing files in RAW_DIR: {missing}"

RAW_DIR

PosixPath('/Users/manuel.orozco/Desktop/SALES ANALYTICS DASHBOARD/01-data/01-raw')

### CSV ingestion note (order_reviews)

`order_reviews_dataset.csv` contains free-text fields and malformed rows (e.g., unescaped commas and quoted newlines).
To ensure reproducible ingestion in DuckDB, we load it with:
- `parallel=false` (compatibility with quoted newlines)
- `null_padding=true` (fills missing columns with NULL)
- `all_varchar=true` (avoids type inference failures caused by malformed rows)

Types are enforced downstream in SQL models using `try_cast(...)`.

In [5]:
# Create/replace staging tables directly from CSVs
con.execute(f"""
create or replace table stg_orders as
select * from read_csv_auto('{(RAW_DIR / expected_files["orders"]).as_posix()}');
""")

con.execute(f"""
create or replace table stg_customers as
select * from read_csv_auto('{(RAW_DIR / expected_files["customers"]).as_posix()}');
""")

con.execute(f"""
create or replace table stg_order_items as
select * from read_csv_auto('{(RAW_DIR / expected_files["order_items"]).as_posix()}');
""")

con.execute(f"""
create or replace table stg_order_payments as
select * from read_csv_auto('{(RAW_DIR / expected_files["order_payments"]).as_posix()}');
""")

reviews_path = (RAW_DIR / expected_files["order_reviews"]).as_posix()
con.execute(f"""
create or replace table stg_order_reviews as
select * from read_csv(
    '{reviews_path}',
    header=true,
    auto_detect=true,
    strict_mode=false,
    null_padding=true,
    parallel=false,
    all_varchar=true
);
""")

con.execute(f"""
create or replace table stg_products as
select * from read_csv_auto('{(RAW_DIR / expected_files["products"]).as_posix()}');
""")

con.execute("show tables;").fetchdf()

,name
0,dim_customers
1,dim_date
2,dim_products
3,fact_order_items
4,fact_order_payments
5,fact_order_reviews
6,fact_orders
7,mart_cohort_unit_economics
8,mart_customer_ltv_summary
9,mart_monthly_business_snapshot


In [6]:
con.execute("""
select 'stg_orders' as table_name, count(*) as rows from stg_orders
union all select 'stg_customers', count(*) from stg_customers
union all select 'stg_order_items', count (*) from stg_order_items
union all select 'stg_order_payments', count (*) from stg_order_payments
union all select 'stg_order_reviews', count (*) from stg_order_reviews
union all select 'stg_products', count (*) from stg_products
order by table_name;
""").fetchdf()

,table_name,rows
0,stg_customers,99441
1,stg_order_items,112650
2,stg_order_payments,103886
3,stg_order_reviews,100002
4,stg_orders,99441
5,stg_products,32340


## Build star schema from SQL scripts (`03-sql/`)

We execute SQL scripts from `03-sql/schema/` and `03-sql/models/`.
To enforce execution order, scripts should use numeric prefixes (e.g., `01_*.sql`, `02_*.sql`).

Recommended order:
1) `schema/` (DDL)
2) `models/` (SELECT/INSERT builds)

In [7]:
SQL_ROOT = PROJECT_ROOT / "03-sql"
SCHEMA_DIR = SQL_ROOT / "schema"
MODELS_DIR = SQL_ROOT / "models"
MARTS_DIR = SQL_ROOT / "marts"

def run_sql_folder(folder: Path) -> list[Path]:
    if not folder.exists():
        print(f"Folder not found, skipping: {folder}")
        return[]

    sql_files = sorted(folder.glob("*.sql")) #relies on numeric prefixes for order
    print(f"Running {len(sql_files)} scripts from: {folder}")

    for f in sql_files:
        try:
            print(f" -> {f.name}")
            sql = f.read_text(encoding="utf-8")
            con.execute(sql)
        except Exception as e:
            raise RuntimeError(f"Failed executing {f.name}: {e}") from e
            
    return sql_files

ran_schema = run_sql_folder(SCHEMA_DIR)
ran_models = run_sql_folder(MODELS_DIR)
ran_marts = run_sql_folder(MARTS_DIR)

con.execute("show tables;").fetchdf()

Running 0 scripts from: /Users/manuel.orozco/Desktop/SALES ANALYTICS DASHBOARD/03-sql/schema
Running 7 scripts from: /Users/manuel.orozco/Desktop/SALES ANALYTICS DASHBOARD/03-sql/models
 -> 01_dim_date.sql
 -> 02_dim_customers.sql
 -> 03_dim_products.sql
 -> 04_fact_orders.sql
 -> 05_fact_order_items.sql
 -> 06_fact_order_payments.sql
 -> 07_fact_order_reviews.sql
Running 3 scripts from: /Users/manuel.orozco/Desktop/SALES ANALYTICS DASHBOARD/03-sql/marts
 -> 01_mart_cohort_unit_economics.sql
 -> 02_mart_monthly_business_snapshot.sql
 -> 03_mart_customer_ltv_summary.sql


,name
0,dim_customers
1,dim_date
2,dim_products
3,fact_order_items
4,fact_order_payments
5,fact_order_reviews
6,fact_orders
7,mart_cohort_unit_economics
8,mart_customer_ltv_summary
9,mart_monthly_business_snapshot


In [8]:
expected_star_tables = [
    "dim_date",
    "dim_customers",
    "dim_products",
    "fact_orders",
    "fact_order_items",
    "fact_order_payments",
    "fact_order_reviews",
]
present = set(con.execute("show tables;").fetchdf()["name"].tolist())
missing_star = [t for t in expected_star_tables if t not in present]

missing_star

[]

In [9]:
# Row counts (only valid if the tables exist)
if missing_star:
    print("SKIP row counts. Missing star tables:", missing_star)
else:
    con.execute("""
    select 'dim_date' as table_name, count(*) as rows from dim_date
    union all select 'dim_customers', count (*) from dim_customers
    union all select 'dim_products', count (*) from dim_products
    union all select 'fact_order_items', count (*) from fact_order_items
    order by table_name;
    """).fetchdf()

## Validation suite (Phase 3)

This section validates that the materialized tables in DuckDB comply with the **model contracts** defined for the star schema.

### Guiding Principles

- All transformation logic lives in SQL (`03-sql/models`).
- The notebook is responsible only for **orchestration and validation**.
- No business logic should exist in Python.
- Every table must explicitly declare and satisfy its grain.

### Phase 3 Definition of Done

The following validations must pass for each dimension and fact table:

1) **Table existence** - expected dimension and fact tables are present.
2) **Row count sanity checks** - basic volume validation.
3) **Explicit grain validation** - uniqueness of the declated primary key.
4) **Null checks on primary keys** - no nulls allowed in PK fields.
5) **Referential integrity checks** (for fact tables) - all foreign keys must match existing dimension keys.

These validations must be executed **after** running:
ran_models = run_sql_folder(MODELS_DIR)

In [10]:
# ===============================
# VALIDATION - dim_date
# Contract:
#   Grain: 1 row per calendar date
#   PK: date_key must be unique and non-null
#   Range: continuous between min/max purchase dates in stg_orders
# ===============================

# 1) Row count (sanity check)
rows = con.execute("select count(*) as rows from dim_date;").fetchone()[0]


# 2) Date range (should align with stg_orders purchase timestamp bounds)
min_date, max_date = con.execute("""
select
    min(date(date)) as min_date, 
    max(date(date)) as max_date 
from dim_date;
""").fetchone()


# 3) PK uniqueness (grain contract)
key_unique = con.execute("""
select
    count(*) = count(distinct date_key)
from dim_date;
""").fetchone()[0]


# 4) PK null check (hard fail condition)
null_pk = con.execute("""
select 
    count(*) as null_date_key
from dim_date where date_key is null;
""").fetchone()[0]

# 5) Bounds alignment vs staging (strong check)
stg_min, stg_max = con.execute("""
select
    min(date(order_purchase_timestamp)),
    max(date(order_purchase_timestamp))
from stg_orders;
""").fetchone()

print("DIM_DATE VALIDATION")
print("-------------------")
print(f"Row count: {rows}")
print(f"Date range: {min_date} -> {max_date}")
print(f"PK unique: {key_unique}")
print(f"Null PKs: {null_pk}")
print(f"Staging bounds (stg_orders): {stg_min} -> {stg_max}")

dim_date_passed = (
    (key_unique is True)
    and (null_pk == 0)
    and(min_date == stg_min)
    and(max_date == stg_max)
)

print(f"RESULT: {'PASS' if dim_date_passed else 'FAIL'}")

DIM_DATE VALIDATION
-------------------
Row count: 774
Date range: 2016-09-04 -> 2018-10-17
PK unique: True
Null PKs: 0
Staging bounds (stg_orders): 2016-09-04 -> 2018-10-17
RESULT: PASS


In [11]:
# ===============================
# VALIDATION - dim_customers
# Contract:
#   Grain: 1 row per customer_id
#   PK: customer_id must be unique and non-null
# ===============================

# 1) Row count (sanity check)
rows = con.execute("select count(*) as rows from dim_customers;").fetchone()[0]


# 2) PK uniqueness (grain contract)
key_unique = con.execute("""
select 
    count(*) = count(distinct customer_id)
from dim_customers;
""").fetchone()[0]


# 3) PK null check (hard fail condition)
null_pk = con.execute("""
select 
    count(*) as null_customer_id
from dim_customers where customer_id is null;
""").fetchone()[0]

print("DIM_CUSTOMERS VALIDATION")
print("------------------------")
print(f"Row count: {rows}")
print(f"PK unique (customer_id): {key_unique}")
print(f"Null PKs (customer_id): {null_pk}")

passed = (key_unique is True) and (null_pk ==0)
print(f"RESULT: {'PASS' if passed else 'FAIL'}")

DIM_CUSTOMERS VALIDATION
------------------------
Row count: 99441
PK unique (customer_id): True
Null PKs (customer_id): 0
RESULT: PASS


In [12]:
# ===============================
# VALIDATION - dim products
# Contract:
#   Grain: 1 row per product_id
#   PK: product_id must be unique and non-null
# ===============================

# 1) Row count (sanity check)
rows = con.execute("select count(*) as rows from dim_products;").fetchone()[0]


# 2) PK uniqueness (grain contract)
key_unique = con.execute("""
select
    count(*) = count(distinct product_id)
from dim_products;
""").fetchone()[0]


# 3) PK null check (hard fail condition)
null_pk = con.execute("""
select
    count(*) as null_product_id
from dim_products where product_id is null;
""").fetchone()[0]


print("DIM_PRODUCTS VALIDATION")
print("-----------------------")
print(f"Row count: {rows}")
print(f"PK unique (product_id): {key_unique}")
print(f"Null PKs (product_id): {null_pk}")

passed = (key_unique is True) and (null_pk == 0)
print(f"RESULT: {'PASS' if passed else 'FAIL'}")

DIM_PRODUCTS VALIDATION
-----------------------
Row count: 32951
PK unique (product_id): True
Null PKs (product_id): 0
RESULT: PASS


In [13]:
# ===============================
# VALIDATION - fact_orders
# Contract:
#   Grain: 1 row per order_id
#   Keys:
#     - order_id unique and non-null
#     - customer_id non-null and exists in dim_customers
#     - purchase_date_key non-null and exists in dim_date
# ===============================

# 1) Row count (sanity check)
rows = con.execute("select count(*) from fact_orders;").fetchone()[0]

# 2) Grain check (order_id must be unique)
grain_ok = con.execute("""
select count(*) = count(distinct order_id)
from fact_orders;
""").fetchone()[0]

# 3) Null checks (hard fail conditions)
null_order_id = con.execute("select count(*) from fact_orders where order_id is null;").fetchone()[0]
null_customer_id = con.execute("select count(*) from fact_orders where customer_id is null;").fetchone()[0]
null_date_key = con.execute("select count(*) from fact_orders where purchase_date_key is null;").fetchone()[0]

# 4) Referential integrity (orphans should be 0)
orphans_customers = con.execute("""
select count(*)
from fact_orders f
left join dim_customers c on c.customer_id = f.customer_id
where c.customer_id is null;
""").fetchone()[0]

orphans_dates = con.execute("""
select count(*)
from fact_orders f
left join dim_date d on d.date_key = f.purchase_date_key
where d.date_key is null;
""").fetchone()[0]

print("FACT_ORDERS VALIDATION")
print("----------------------")
print(f"Row count: {rows}")
print(f"Grain OK (order_id unique): {grain_ok}")

print("Null key checks:")
print(f"  null_order_id: {null_order_id}")
print(f"  null_customer_id: {null_customer_id}")
print(f"  null_purchase_date_key: {null_date_key}")

print("Referential integrity (orphans):")
print(f"  orphans_customers: {orphans_customers}")
print(f"  orphans_dates: {orphans_dates}")

passed = (
    (grain_ok is True)
    and (null_order_id == 0)
    and (null_customer_id == 0)
    and (null_date_key == 0)
    and (orphans_customers == 0)
    and (orphans_dates == 0)
)
print(f"RESULT: {'PASS' if passed else 'FAIL'}")

FACT_ORDERS VALIDATION
----------------------
Row count: 99441
Grain OK (order_id unique): True
Null key checks:
  null_order_id: 0
  null_customer_id: 0
  null_purchase_date_key: 0
Referential integrity (orphans):
  orphans_customers: 0
  orphans_dates: 0
RESULT: PASS


In [14]:
# ===============================
# VALIDATION - fact_order_items
# Contract:
#   Grain: 1 row per (order_id, order_item_id)
#   Keys:
#     - order_id not null
#     - order_item_id not null
#     - customer_id not null (exists in dim_customers)
#     - product_id not null (exists in dim_products)
#     - pruchase_date_key not null (exists in dim_date)
#   Referential integrity:
#     - order_id must exist in fact_orders (hub)
# ===============================

# 1) Row count (sanity check)
rows = con.execute("select count(*) as rows from fact_order_items;").fetchone()[0]


# 2) Grain check (order_id, order_item_id must be unique)
grain_ok = con.execute("""
select 
    count(*) = count(distinct (order_id, order_item_id))
from fact_order_items;
""").fetchone()[0]


# 3) Null checks (hard fail conditions)
null_order_id = con.execute("""
select count(*) from fact_order_items where order_id is null;
""").fetchone()[0]

null_order_item_id = con.execute("""
select count(*) from fact_order_items where order_item_id is null;
""").fetchone()[0]

null_customer_id = con.execute("""
select count(*) from fact_order_items where customer_id is null;
""").fetchone()[0]

null_product_id = con.execute("""
select count(*) from fact_order_items where product_id is null;
""").fetchone()[0]

null_date_key = con.execute("""
select count(*) from fact_order_items where date_key is null;
""").fetchone()[0]


# 4) Referential integrity (orphans should be 0)
orphans_dates = con.execute("""
select count(*)
from fact_order_items f
left join dim_date d on d.date_key = f.date_key
where d.date_key is null;
""").fetchone()[0]

orphans_customers = con.execute("""
select count(*)
from fact_order_items f
left join dim_customers c on c.customer_id = f.customer_id
where c.customer_id is null;
""").fetchone()[0]

orphans_products = con.execute("""
select count(*)
from fact_order_items f
left join dim_products p on p.product_id = f.product_id
where p.product_id is null;
""").fetchone()[0]

orphans_orders = con.execute("""
select count(*)
from fact_order_items f
left join fact_orders o on o.order_id = f.order_id
where o.order_id is null;
""").fetchone()[0]


print("FACT_ORDER_ITEMS VALIDATION")
print("---------------------------")
print(f"Row count: {rows}")
print(f"Grain OK (order_id, order_item_id unique): {grain_ok}")

print("Null key checks:")
print(f"  null_order_id: {null_order_id}")
print(f"  null_order_item_id: {null_order_item_id}")
print(f"  null_customer_id: {null_customer_id}")
print(f"  null_product_id: {null_product_id}")
print(f"  null_date_key: {null_date_key}")

print("Referential integrity (orphans):")
print(f"  orphans_orders: {orphans_orders}")
print(f"  orphans_customers: {orphans_customers}")
print(f"  orphans_products: {orphans_products}")
print(f"  orphans_date_keys: {orphans_dates}")

passed = (
    (grain_ok is True)
    and (null_order_id == 0)
    and (null_order_item_id == 0)
    and (null_customer_id == 0)
    and (null_product_id == 0)
    and (null_date_key == 0)
    and (orphans_dates == 0)
    and (orphans_customers == 0)
    and (orphans_products == 0)
    and (orphans_orders == 0)
)
print(f"RESULT: {'PASS' if passed else 'FAIL'}")

FACT_ORDER_ITEMS VALIDATION
---------------------------
Row count: 112650
Grain OK (order_id, order_item_id unique): True
Null key checks:
  null_order_id: 0
  null_order_item_id: 0
  null_customer_id: 0
  null_product_id: 0
  null_date_key: 0
Referential integrity (orphans):
  orphans_orders: 0
  orphans_customers: 0
  orphans_products: 0
  orphans_date_keys: 0
RESULT: PASS


In [15]:
con.execute("""
select f.product_id, count(*) as rows
from fact_order_items f
left join dim_products p on p.product_id = f.product_id
where p.product_id is null
group by 1
order by rows desc
limit 20;
""").fetchdf()

,product_id,rows


In [16]:
con.execute("""
select
  (select count(distinct product_id) from stg_order_items) as distinct_products_in_items,
  (select count(distinct product_id) from stg_products) as distinct_products_in_catalog,
  (select count(distinct oi.product_id)
   from stg_order_items oi
   left join stg_products p on p.product_id = oi.product_id
   where p.product_id is null
  ) as distinct_missing_in_catalog;
""").fetchdf()

,distinct_products_in_items,distinct_products_in_catalog,distinct_missing_in_catalog
0,32951,32340,611


In [17]:
# ===============================
# VALIDATION - fact_order_payments
# Contract:
#   Grain: 1 row per (order_id, payment_sequential)
#   Keys:
#     - order_id not null (exists in fact_orders)
#     - payment_sequential not null
#     - customer_id not null (exists in dim_customers)
#     - purchase_date_key not null (exists in dim_date)
# ===============================

# 1) Row count (sanity check)
rows = con.execute("select count(*) from fact_order_payments;").fetchone()[0]


# 2) Grain check (order_id, payment_sequential must be unique)
grain_ok = con.execute("""
select count(*) = count(distinct (order_id, payment_sequential))
from fact_order_payments;
""").fetchone()[0]


# 3) Null checks (hard fail conditions)
null_order_id = con.execute("select count(*) from fact_order_payments where order_id is null;").fetchone()[0]
null_payment_sequential = con.execute("select count(*) from fact_order_payments where payment_sequential is null;").fetchone()[0]
null_customer_id = con.execute("select count(*) from fact_order_payments where customer_id is null;").fetchone()[0]
null_purchase_date_key = con.execute("select count(*) from fact_order_payments where purchase_date_key is null;").fetchone()[0]


# 4) Referential integrity (orphans should be 0)
orphans_orders = con.execute("""
select count(*)
from fact_order_payments f
left join fact_orders o on o.order_id = f.order_id
where o.order_id is null;
""").fetchone()[0]

orphans_customers = con.execute("""
select count(*)
from fact_order_payments f
left join dim_customers c on c.customer_id = f.customer_id
where c.customer_id is null;
""").fetchone()[0]

orphans_dates = con.execute("""
select count(*)
from fact_order_payments f
left join dim_date d on d.date_key = f.purchase_date_key
where d.date_key is null;
""").fetchone()[0]


print("FACT_ORDER_PAYMENTS VALIDATION")
print("------------------------------")
print(f"Row count: {rows}")
print(f"Grain OK (order_id, payment_sequential unique): {grain_ok}")

print("Null key checks:")
print(f"  null_order_id: {null_order_id}")
print(f"  null_payment_sequential: {null_payment_sequential}")
print(f"  null_customer_id: {null_customer_id}")
print(f"  null_purchase_date_key: {null_purchase_date_key}")

print("Referential integrity (orphans):")
print(f"  orphans_orders: {orphans_orders}")
print(f"  orphans_customers: {orphans_customers}")
print(f"  orphans_dates: {orphans_dates}")

passed = (
    (grain_ok is True)
    and (null_order_id == 0)
    and (null_payment_sequential == 0)
    and (null_customer_id == 0)
    and (null_purchase_date_key == 0)
    and (orphans_orders == 0)
    and (orphans_customers == 0)
    and (orphans_dates == 0)
)

print(f"RESULT: {'PASS' if passed else 'FAIL'}")

FACT_ORDER_PAYMENTS VALIDATION
------------------------------
Row count: 103886
Grain OK (order_id, payment_sequential unique): True
Null key checks:
  null_order_id: 0
  null_payment_sequential: 0
  null_customer_id: 0
  null_purchase_date_key: 0
Referential integrity (orphans):
  orphans_orders: 0
  orphans_customers: 0
  orphans_dates: 0
RESULT: PASS


In [18]:
# ===============================
# VALIDATION - fact_order_reviews
# Contract:
#   Grain: 1 row per review_id
#   Keys:
#     - review_id not null and unique
#     - order_id not null (exists in fact_orders)
#     - customer_id not null (exists in dim_customers)
#     - purchase_date_key not null (exists in dim_date)
# ===============================

# 1) Row count (sanity check)
rows = con.execute("select count(*) from fact_order_reviews;").fetchone()[0]


# 2) Grain check (review_id, order_id must be unique)
grain_ok = con.execute("""
select count(*) = count(distinct (review_id, order_id))
from fact_order_reviews;
""").fetchone()[0]


# 3) Null checks (hard fail conditions)
null_review_id = con.execute("select count(*) from fact_order_reviews where review_id is null;").fetchone()[0]
null_order_id = con.execute("select count(*) from fact_order_reviews where order_id is null;").fetchone()[0]
null_customer_id = con.execute("select count(*) from fact_order_reviews where customer_id is null;").fetchone()[0]
null_purchase_date_key = con.execute("select count(*) from fact_order_reviews where purchase_date_key is null;").fetchone()[0]


# 4) Referential integrity (orphans should be 0)
orphans_orders = con.execute("""
select count(*)
from fact_order_reviews f
left join fact_orders o on o.order_id = f.order_id
where o.order_id is null;
""").fetchone()[0]

orphans_customers = con.execute("""
select count(*)
from fact_order_reviews f
left join dim_customers c on c.customer_id = f.customer_id
where c.customer_id is null;
""").fetchone()[0]

orphans_dates = con.execute("""
select count(*)
from fact_order_reviews f
left join dim_date d on d.date_key = f.purchase_date_key
where d.date_key is null;
""").fetchone()[0]


print("FACT_ORDER_REVIEWS VALIDATION")
print("-----------------------------")
print(f"Row count: {rows}")
print(f"Grain OK (review_id, order_id unique): {grain_ok}")

print("Null key checks:")
print(f"  null_review_id: {null_review_id}")
print(f"  null_order_id: {null_order_id}")
print(f"  null_customer_id: {null_customer_id}")
print(f"  null_purchase_date_key: {null_purchase_date_key}")

print("Referential integrity (orphans):")
print(f"  orphans_orders: {orphans_orders}")
print(f"  orphans_customers: {orphans_customers}")
print(f"  orphans_dates: {orphans_dates}")

passed = (
    (grain_ok is True)
    and (null_review_id == 0)
    and (null_order_id == 0)
    and (null_customer_id == 0)
    and (null_purchase_date_key == 0)
    and (orphans_orders == 0)
    and (orphans_customers == 0)
    and (orphans_dates == 0)
)

print(f"RESULT: {'PASS' if passed else 'FAIL'}")

FACT_ORDER_REVIEWS VALIDATION
-----------------------------
Row count: 100000
Grain OK (review_id, order_id unique): True
Null key checks:
  null_review_id: 0
  null_order_id: 0
  null_customer_id: 0
  null_purchase_date_key: 0
Referential integrity (orphans):
  orphans_orders: 0
  orphans_customers: 0
  orphans_dates: 0
RESULT: PASS


In [19]:
# ===============================
# RI CHECKS - facts -> fact_orders
# Contract:
#   All facts must anchor to fact_orders by order_id (no orphan orders)
# ===============================

# 1) Orphans: fact_order_items -> fact_orders
orphans_items_orders = con.execute("""
select count(*)
from fact_order_items f
left join fact_orders o on o.order_id = f.order_id
where o.order_id is null;
""").fetchone()[0]


# 2) Orphans: fact_order_payments -> fact_orders
orphans_payments_orders = con.execute("""
select count(*)
from fact_order_payments f
left join fact_orders o on o.order_id = f.order_id
where o.order_id is null;
""").fetchone()[0]


# 3) Orphans: fact_order_reviews -> fact_orders
orphans_reviews_orders = con.execute("""
select count(*)
from fact_order_reviews f
left join fact_orders o on o.order_id = f.order_id
where o.order_id is null;
""").fetchone()[0]


print("RI CHECKS - FACTS -> FACT_ORDERS")
print("--------------------------------")
print(f"orphans_items_orders: {orphans_items_orders}")
print(f"orphans_payments_orders: {orphans_payments_orders}")
print(f"orphans_reviews_orders: {orphans_reviews_orders}")

passed = (
    (orphans_items_orders == 0)
    and (orphans_payments_orders == 0)
    and (orphans_reviews_orders == 0)
)

print(f"RESULT: {'PASS' if passed else 'FAIL'}")

RI CHECKS - FACTS -> FACT_ORDERS
--------------------------------
orphans_items_orders: 0
orphans_payments_orders: 0
orphans_reviews_orders: 0
RESULT: PASS


In [20]:
SQL_ROOT = PROJECT_ROOT / "03-sql"
SCHEMA_DIR = SQL_ROOT / "schema"
MODELS_DIR = SQL_ROOT / "models"
MARTS_DIR = SQL_ROOT / "marts"

sorted([p.name for p in SCHEMA_DIR.glob("*.sql")]), sorted([p.name for p in MODELS_DIR.glob("*.sql")]), sorted([p.name for p in MARTS_DIR.glob("*.sql")])

([],
 ['01_dim_date.sql',
  '02_dim_customers.sql',
  '03_dim_products.sql',
  '04_fact_orders.sql',
  '05_fact_order_items.sql',
  '06_fact_order_payments.sql',
  '07_fact_order_reviews.sql'],
 ['01_mart_cohort_unit_economics.sql',
  '02_mart_monthly_business_snapshot.sql',
  '03_mart_customer_ltv_summary.sql'])

In [21]:
# ===============================
# VALIDATION — mart_cohort_unit_economics
# Contract:
#   Grain: 1 row per (cohort_month, months_since_first_purchase)
#   cohort_month: month of first purchase
#   months_since_first_purchase: 0,1,2,... relative to cohort start
# ===============================

# 1) Row count (sanity check)
rows = con.execute("""
select count(*) 
from mart_cohort_unit_economics;
""").fetchone()[0]


# 2) Grain check (cohort_month + months_since_first_purchase must be unique)
grain_ok = con.execute("""
select count(*) = count(distinct (cohort_month, months_since_first_purchase))
from mart_cohort_unit_economics;
""").fetchone()[0]


# 3) Null checks (hard fail conditions)
null_cohort_month = con.execute("""
select count(*) 
from mart_cohort_unit_economics
where cohort_month is null;
""").fetchone()[0]

null_age = con.execute("""
select count(*) 
from mart_cohort_unit_economics
where months_since_first_purchase is null;
""").fetchone()[0]


# 4) Sanity checks (business logic consistency)

# Cohort age should never be negative
negative_age = con.execute("""
select count(*) 
from mart_cohort_unit_economics
where months_since_first_purchase < 0;
""").fetchone()[0]

# Retention rate must be between 0 and 1 (if defined)
bad_retention = con.execute("""
select count(*)
from mart_cohort_unit_economics
where retention_rate is not null
  and (retention_rate < -0.000001 or retention_rate > 1.000001);
""").fetchone()[0]

# Revenue should not be negative (your column is named "revenue")
negative_revenue = con.execute("""
select count(*)
from mart_cohort_unit_economics
where revenue < 0;
""").fetchone()[0]


print("MART_COHORT_UNIT_ECONOMICS VALIDATION")
print("-------------------------------------")
print(f"Row count: {rows}")
print(f"Grain OK (cohort_month, months_since_first_purchase unique): {grain_ok}")

print("Null checks:")
print(f"  null_cohort_month: {null_cohort_month}")
print(f"  null_months_since_first_purchase: {null_age}")

print("Sanity checks:")
print(f"  negative_age_rows: {negative_age}")
print(f"  bad_retention_rate_rows: {bad_retention}")
print(f"  negative_revenue_rows: {negative_revenue}")

passed = (
    (grain_ok is True)
    and (null_cohort_month == 0)
    and (null_age == 0)
    and (negative_age == 0)
    and (bad_retention == 0)
    and (negative_revenue == 0)
)

print(f"RESULT: {'PASS' if passed else 'FAIL'}")

MART_COHORT_UNIT_ECONOMICS VALIDATION
-------------------------------------
Row count: 24
Grain OK (cohort_month, months_since_first_purchase unique): True
Null checks:
  null_cohort_month: 0
  null_months_since_first_purchase: 0
Sanity checks:
  negative_age_rows: 0
  bad_retention_rate_rows: 0
  negative_revenue_rows: 0
RESULT: PASS


In [22]:
# ===============================
# VALIDATION — mart_monthly_business_snapshot
# Contract:
#   Grain: 1 row per purchase_month
# ===============================

# 1) Row count (sanity check)
rows = con.execute("select count(*) from mart_monthly_business_snapshot;").fetchone()[0]

# 2) Grain check (purchase_month must be unique)
grain_ok = con.execute("""
select count(*) = count(distinct purchase_month)
from mart_monthly_business_snapshot;
""").fetchone()[0]

# 3) Null checks (hard fail conditions)
null_month = con.execute("""
select count(*) from mart_monthly_business_snapshot where purchase_month is null;
""").fetchone()[0]

# 4) Metric sanity (rates should be 0..1 where defined)
bad_ontime = con.execute("""
select count(*)
from mart_monthly_business_snapshot
where on_time_delivery_rate is not null
  and (on_time_delivery_rate < -0.000001 or on_time_delivery_rate > 1.000001);
""").fetchone()[0]

print("MART_MONTHLY_BUSINESS_SNAPSHOT VALIDATION")
print("----------------------------------------")
print(f"Row count: {rows}")
print(f"Grain OK (purchase_month unique): {grain_ok}")
print("Null checks:")
print(f"  null_purchase_month: {null_month}")
print("Sanity checks:")
print(f"  bad_on_time_delivery_rate_rows: {bad_ontime}")

passed = (
    (grain_ok is True)
    and (null_month == 0)
    and (bad_ontime == 0)
)

print(f"RESULT: {'PASS' if passed else 'FAIL'}")

MART_MONTHLY_BUSINESS_SNAPSHOT VALIDATION
----------------------------------------
Row count: 24
Grain OK (purchase_month unique): True
Null checks:
  null_purchase_month: 0
Sanity checks:
  bad_on_time_delivery_rate_rows: 0
RESULT: PASS


In [23]:
# ===============================
# VALIDATION — mart_customer_ltv_summary
# Contract:
#   Grain: 1 row per customer_id
# ===============================

# 1) Row count (sanity check)
rows = con.execute("select count(*) from mart_customer_ltv_summary;").fetchone()[0]

# 2) Grain check (customer_id must be unique)
grain_ok = con.execute("""
select count(*) = count(distinct customer_id)
from mart_customer_ltv_summary;
""").fetchone()[0]

# 3) Null checks
null_customer_id = con.execute("""
select count(*) from mart_customer_ltv_summary where customer_id is null;
""").fetchone()[0]

# 4) Business sanity
negative_revenue = con.execute("""
select count(*)
from mart_customer_ltv_summary
where total_revenue < 0;
""").fetchone()[0]

print("MART_CUSTOMER_LTV_SUMMARY VALIDATION")
print("-----------------------------------")
print(f"Row count: {rows}")
print(f"Grain OK (customer_id unique): {grain_ok}")
print("Null checks:")
print(f"  null_customer_id: {null_customer_id}")
print("Sanity checks:")
print(f"  negative_revenue_rows: {negative_revenue}")

passed = (
    (grain_ok is True)
    and (null_customer_id == 0)
    and (negative_revenue == 0)
)

print(f"RESULT: {'PASS' if passed else 'FAIL'}")

MART_CUSTOMER_LTV_SUMMARY VALIDATION
-----------------------------------
Row count: 98816
Grain OK (customer_id unique): True
Null checks:
  null_customer_id: 0
Sanity checks:
  negative_revenue_rows: 0
RESULT: PASS


In [24]:
con.execute("show tables;").fetchdf()

,name
0,dim_customers
1,dim_date
2,dim_products
3,fact_order_items
4,fact_order_payments
5,fact_order_reviews
6,fact_orders
7,mart_cohort_unit_economics
8,mart_customer_ltv_summary
9,mart_monthly_business_snapshot


## End of Phase 3 - SQL Materialization Complete (DuckDB)

The analytical model has been fully materialized in DuckDB using deterministic SQL scripts stored in `03-sql/`.

All transformations now live exclusively in SQL.
The notebook is responsible only for orchestration and validation.

### Materialized Tables

Dimensions:
1. `03-sql/01_dim_date.sql`
2. `03-sql/02_dim_customers.sql`
3. `03-sql/03_dim_products.sql`

Facts
4. `03-sql/04_fact_orders.sql`
5. `03-sql/05_fact_order_items.sql`
6. `03-sql/06_fact_order_payments.sql`
7. `03-sql/07_fact_order_reviews.sql`

### Model Contracts (Explicit Grain)

- `dim_date` → 1 row per calendar date  
- `dim_customers` → 1 row per `customer_id`  
- `dim_products` → 1 row per `product_id` (including backfilled keys from order items)

- `fact_orders` → 1 row per `order_id`  
- `fact_order_items` → 1 row per (`order_id`, `order_item_id`)  
- `fact_order_payments` → 1 row per (`order_id`, `payment_sequential`)  
- `fact_order_reviews` → 1 row per (`review_id`, `order_id`)

### Validation Status

Each table has been validated for:

- Row count (sanity check)
- Grain uniqueness
- Null key constraints
- Referential integrity (no orphan keys)
- Cross-fact anchoring to `fact_orders`

All validation checks pass.

The model is now:

- Deterministic (script-ordered execution)
- Reproducible (full rebuild via SQL runner)
- Referentially consistent
- Explicitly documented at the grain level

Phase 3 is complete.